# Go2 RL 보행 학습 (Colab)
1. 셀 1~4: GPU 확인, Drive 마운트, 설치, **T4 속도 측정**(공식 Genesis Go2 예제, 이 저장소와 무관)
2. 셀 5: 이 저장소 클론 후 학습. Drive에 체크포인트를 저장하고 재연결 시 `--resume`

> 셀 5는 RL 코드(`module1_locomotion/rl_training`)가 GitHub에 push되어 있어야 한다. 아래 `BRANCH`가 그 브랜치를 가리켜야 한다.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Colab은 자체 CUDA torch를 가지므로 requirements.txt의 torch 줄은 건너뛴다
REQ = "https://raw.githubusercontent.com/codyssey-dt-doosan/32-dt-doosan-quadruped/main/module1_locomotion/rl_training/requirements.txt"
!wget -q -O /content/req_all.txt $REQ
!grep -v '^torch==' /content/req_all.txt > /content/req.txt
!pip install -q -r /content/req.txt

## 속도 측정 (T4)
Genesis v1.5.0의 공식 `go2_train.py`로 101 iter를 돌려 총 시간과 step/s를 본다. 비교 기준: Mac 약 5.7만 step/s(공식 예제). 이 저장소 env는 물리 4 substep이라 Mac에서 3.1만 step/s였다.

In [ ]:
import os, subprocess
BASE = "https://raw.githubusercontent.com/Genesis-Embodied-AI/genesis-world/v1.5.0/examples/locomotion"
os.makedirs("/content/speed", exist_ok=True)
for f in ("go2_env.py", "go2_train.py"):
    subprocess.run(["wget", "-q", "-O", f"/content/speed/{f}", f"{BASE}/{f}"], check=True)
r = subprocess.run(
    "cd /content/speed && python go2_train.py -e t4_speed -b 4096 --max-iterations 101",
    shell=True, capture_output=True, text=True)
out = r.stdout + r.stderr
print(out[-1500:] if r.returncode else "")
for line in out.splitlines():
    if any(k in line.lower() for k in ("total time", "steps/s", "step/s", "fps", "collection time")):
        print(line)

## 학습
`BRANCH`를 RL 코드가 있는 브랜치로 바꾼다(기본 `main`). 로그·체크포인트는 Drive `rl_go2/<EXP>`에 쌓이고, 이미 `model_*.pt`가 있으면 `--resume`으로 이어서 학습한다.

In [ ]:
BRANCH = "main"
EXP = "walk_v1"
BATCH = 4096
MAX_ITER = 3000
LOG_DIR = "/content/drive/MyDrive/rl_go2"  # train.py가 뒤에 <EXP>를 붙인다

In [ ]:
import glob, os
REPO = "/content/32-dt-doosan-quadruped"
if not os.path.isdir(REPO):
    !git clone -b $BRANCH https://github.com/codyssey-dt-doosan/32-dt-doosan-quadruped.git $REPO
%cd $REPO/module1_locomotion/rl_training
resume = "--resume" if glob.glob(f"{LOG_DIR}/{EXP}/model_*.pt") else ""
print("resume:", bool(resume))
!python train.py -e $EXP -b $BATCH --max-iterations $MAX_ITER --log-dir $LOG_DIR --save-interval 50 $resume